# Merge EPA Water Quality Measurements with Climate Data

In [1]:
import pandas as pd
import os

# Load EPA merged data
epa_df = pd.read_csv('../../data/tabular/merged/epa-merged.csv')

# Load Climate data
climate_df = pd.read_csv('../../data/tabular/climate/clean/isu-climate-clean.csv')

# Load Mapping data
mapping_df = pd.read_csv('../../data/tabular/merged/epa-to-climate-station-map.csv')

In [2]:
# 1. Join EPA data with the mapping to get the climate station for each EPA location
mapping_subset = mapping_df[['MonitoringLocationIdentifier', 'climate_station', 'climate_station_name', 'distance_to_climate_station_km']]
epa_mapped = epa_df.merge(mapping_subset, on='MonitoringLocationIdentifier', how='left')

# 2. Extract date from ActivityStartDateTime to merge with climate 'day'
epa_mapped['day'] = pd.to_datetime(epa_mapped['ActivityStartDateTime']).dt.strftime('%Y-%m-%d')

# 3. Merge with climate data on (climate_station, day)
merged_df = epa_mapped.merge(
    climate_df, 
    left_on=['climate_station', 'day'], 
    right_on=['station', 'day'], 
    how='left'
)

# Clean up redundant columns
if 'station' in merged_df.columns:
    merged_df = merged_df.drop(columns=['station'])

In [3]:
print(f"Merged shape: {merged_df.shape}")
merged_df.head()

Merged shape: (59763, 30)


,ActivityIdentifier,MonitoringLocationIdentifier,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode,ResultValueTypeName,ActivityStartDateTime,OrganizationIdentifier,MonitoringLocationName,MonitoringLocationTypeName,...,station_name,doy,gdd_40_86,high,highc,low,lowc,precip,snow,snowd
0,nwisia.01.02400517,USGS-05474000,"Temperature, water",0.0,deg C,Actual,2024-01-01 12:00:00,USGS-IA,"Skunk River at Augusta, IA",Stream,...,BURLINGTON,1.0,0.0,34.0,1.1,28.0,-2.2,0.0,0.0,0.0
1,nwisia.01.02400551,USGS-05451210,Stream width measure,30.0,ft,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,...,IOWA-FALLS,57.0,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0
2,nwisia.01.02400551,USGS-05451210,"Temperature, water",4.7,deg C,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,...,IOWA-FALLS,57.0,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0
3,nwisia.01.02400551,USGS-05451210,"Temperature, air",8.9,deg C,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,...,IOWA-FALLS,57.0,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0
4,nwisia.01.02400551,USGS-05451210,Specific conductance,579.0,uS/cm @25C,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,...,IOWA-FALLS,57.0,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0


In [4]:
# Save the result
output_path = '../../data/tabular/merged/epa-climate-merged.csv'
merged_df.to_csv(output_path, index=False)
print(f"Saved merged data to {output_path}")

Saved merged data to ../../data/tabular/merged/epa-climate-merged.csv
